# ORPHEUS Runtime

The notebook resolves the project from its current directory or ORPHEUS_PROJECT_DIR. GPU 0 is assigned to Qwen3-VL and GPU 1 to YuE.


In [1]:
import os
from pathlib import Path

# Set the path exactly to your cloned Kaggle directory
os.environ['ORPHEUS_PROJECT_DIR'] = '/kaggle/working/ORPHESUS'

def find_project_dir():
    configured = os.environ.get('ORPHEUS_PROJECT_DIR')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    
    for candidate in candidates:
        if (candidate / 'app' / 'server.py').is_file() and (candidate / 'web' / 'index.html').is_file():
            return candidate
            
    raise RuntimeError('ORPHEUS source files are not visible to this kernel. Open the notebook from the project folder or set ORPHEUS_PROJECT_DIR.')

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
os.environ['ORPHEUS_QWEN_GPU'] = '0'
os.environ['ORPHEUS_YUE_GPU'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'ORPHEUS project: {PROJECT_DIR}')


ORPHEUS project: /kaggle/working/ORPHESUS


## Install application and YuE dependencies


In [2]:
!pip install -q -r requirements.txt
!sudo apt-get -qq update && sudo apt-get -qq install -y git-lfs
!git lfs install


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 58.6 MB/s eta 0:00:00:00:0100:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Updated git hooks.
Git LFS initialized.


In [3]:
import subprocess
import sys

yue_dir = PROJECT_DIR / 'YuE'
if not yue_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/multimodal-art-projection/YuE.git', str(yue_dir)], check=True)
codec_dir = yue_dir / 'inference' / 'xcodec_mini_infer'
if not codec_dir.exists():
    subprocess.run(['git', 'clone', 'https://huggingface.co/m-a-p/xcodec_mini_infer', str(codec_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(yue_dir / 'requirements.txt')], check=True)
print('YuE inference:', yue_dir / 'inference' / 'infer.py')


Cloning into '/kaggle/working/ORPHESUS/YuE'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.5 MB/s eta 0:00:00
YuE inference: /kaggle/working/ORPHESUS/YuE/inference/infer.py


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.19.6 which is incompatible.
ray 2.55.1 requires protobuf>=3.20.3, but you have protobuf 3.19.6 which is incompatible.
google-cloud-vision 3.15.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.19.6 which is incompatible.
onnx 1.22.0 requires protobuf>=4.25.1, but you have protobuf 3.19.6 which is incompatible.
google-cloud-videointelligence 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.19.6 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 3.19.6 which is incompatible.
grpc-google-iam-v1 0.

## Verify the two T4 GPUs


In [4]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in the connected kernel.')
if torch.cuda.device_count() < 2:
    raise RuntimeError('ORPHEUS requires two GPUs: Qwen on 0 and YuE on 1.')
for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    memory_gb = torch.cuda.get_device_properties(index).total_memory / 1024**3
    print(f'GPU {index}: {name} ({memory_gb:.1f} GB)')


GPU 0: Tesla T4 (14.6 GB)
GPU 1: Tesla T4 (14.6 GB)


## Load Qwen3-VL on GPU 0

This downloads `Qwen/Qwen3-VL-8B-Instruct` on first use and loads it in 4-bit mode.


In [5]:
from app.qwen_engine import QwenEngine

qwen = QwenEngine()
allocated = torch.cuda.memory_allocated(0) / 1024**3
print(f'Qwen ready on GPU 0; allocated: {allocated:.2f} GB')
del qwen
torch.cuda.empty_cache()


Loading Qwen3-VL on GPU 0...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

FlashAttention unavailable; using the standard attention implementation: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Qwen3-VL loaded successfully.
Qwen ready on GPU 0; allocated: 5.94 GB


## Preflight YuE on GPU 1


In [6]:
from app.yue_engine import YuEEngine

yue = YuEEngine()
assert yue.ready, 'YuE inference script was not found.'
print(f'YuE ready: {yue.infer_script}')
print('YuE will use GPU 1 with two 30-second lyric sessions.')


YuE ready: /kaggle/working/ORPHESUS/YuE/inference/infer.py
YuE will use GPU 1 with two 30-second lyric sessions.


## Start the Flask application


In [9]:
import subprocess
import sys
import time
import requests

server_process = subprocess.Popen([sys.executable, '-B', '-m', 'app.server'], cwd=PROJECT_DIR)
for _ in range(15):
    try:
        health = requests.get('http://127.0.0.1:8000/api/health', timeout=2).json()
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Flask server did not start.')
print(health)


{'qwen': True, 'service': 'ORPHEUS', 'status': 'online', 'success': True, 'yue': True}


127.0.0.1 - - [12/Aug/2026 15:46:48] "GET /api/health HTTP/1.1" 200 -


<frozen runpy>:128: RuntimeWarning: 'app.server' found in sys.modules after import of package 'app', but prior to execution of 'app.server'; this may result in unpredictable behaviour
Address already in use
Port 8000 is in use by another program. Either identify and stop that program, or start the server with a different port.


 * Serving Flask app 'server'
 * Debug mode: off


## End-to-end generation test

Run this final cell to generate a real short song. It downloads YuE checkpoints on first use and can take several minutes on T4 GPUs.


In [10]:
from IPython.display import Audio, display

payload = {
    'text': 'Photosynthesis uses sunlight to convert carbon dioxide and water into glucose and oxygen.',
    'genre': 'uplifting pop',
    'mood': 'energetic',
    'language': 'English',
}
response = requests.post('http://127.0.0.1:8000/api/generate', json=payload, timeout=1800)
response.raise_for_status()
song = response.json()
assert song['success'] and song['audio_url']
audio_response = requests.get(f"http://127.0.0.1:8000{song['audio_url']}", timeout=60)
audio_response.raise_for_status()
assert audio_response.headers['Content-Type'].startswith('audio/')
print(song['title'])
display(Audio(data=audio_response.content, autoplay=False))


Traceback (most recent call last):
  File "/kaggle/working/ORPHESUS/app/server.py", line 65, in generate
    result = get_controller().generate(text=text, genre=genre, mood=mood, language=language)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/ORPHESUS/app/controller.py", line 23, in generate
    song_payload = self.qwen.generate(
                   ^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/ORPHESUS/app/qwen_engine.py", line 121, in generate
    parsed = self._parse_json(raw)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/ORPHESUS/app/qwen_engine.py", line 178, in _parse_json
    return QwenOutput(**data).model_dump()
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pydantic/main.py", line 250, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

HTTPError: 500 Server Error: INTERNAL SERVER ERROR for url: http://127.0.0.1:8000/api/generate